In [ ]:
#libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [ ]:
#data loading

data1 = pd.read_csv('train.csv')
data2 = pd.read_csv('test.csv')

train_df = data1.copy()
test_df = data2.copy()

print("First 5 rows of train_df: \n", train_df.head())

print("First 5 rows of test_df: \n", test_df.head())

In [ ]:
train_df = train_df.drop(columns=['id'], axis=1)

test_id_placeholder = test_df['id']
test_df = test_df.drop(columns=['id'], axis=1)

In [ ]:
#feature engineering

train_df['in_relationship'] = (train_df['marital_status'] == 'Married').astype(int)
test_df['in_relationship'] = (test_df['marital_status'] == 'Married').astype(int)

train_df['has_studied'] = train_df['education_level'].isin(["Bachelor's", "High School", "Master's", "PhD"]).astype(int)
test_df['has_studied'] = test_df['education_level'].isin(["Bachelor's", "High School", "Master's", "PhD"]).astype(int)

train_df['is_employed'] = train_df['employment_status'].isin(['Employed', 'Self-Employed']).astype(int)
test_df['is_employed'] = test_df['employment_status'].isin(['Employed', 'Self-Employed']).astype(int)

train_df['in_debt'] = (train_df['loan_purpose'] == 'Debt consolidation').astype(int)
test_df['in_debt'] = (test_df['loan_purpose'] == 'Debt consolidation').astype(int)

In [ ]:
train_df

In [ ]:
gender_map = {'Male': 0, 'Female': 1, 'Other': 2}

train_df['gender'] = train_df['gender'].map(gender_map)
test_df['gender'] = test_df['gender'].map(gender_map)

In [ ]:
train_df = train_df.drop(columns=['marital_status', 'education_level', 'employment_status', 'loan_purpose'])
test_df = test_df.drop(columns=['marital_status', 'education_level', 'employment_status', 'loan_purpose'])

In [ ]:
from category_encoders import TargetEncoder

encoder = TargetEncoder()

train_df['grade_encoded'] = encoder.fit_transform(
    train_df['grade_subgrade'],
    train_df['loan_paid_back']
)

test_df['grade_encoded'] = encoder.transform(
    test_df['grade_subgrade']
)

In [ ]:
train_df = train_df.drop(columns='grade_subgrade')
test_df = test_df.drop(columns='grade_subgrade')

In [ ]:
target_placeholder = train_df.pop('loan_paid_back')

train_df['loan_paid_back'] = target_placeholder

In [ ]:
num = train_df.select_dtypes(include='number')

In [ ]:
num1 = round(num.corr(), 2)

# Correlation Matrix-Heatmap Plot
mask = np.zeros_like(num1)
mask[np.triu_indices_from(mask)] = True # optional, to hide repeat half of the matrix

f, ax = plt.subplots(figsize=(20, 10))
sns.set_theme(font_scale=1.5) # increase font size

ax = sns.heatmap(num1, mask=mask, annot=True, annot_kws={"size": 12}, linewidths=.5, cmap="coolwarm", fmt=".2f", ax=ax) # round to 2 decimal places
ax.set_title("Correlation matrix", fontsize=20) # add title
plt.show()


In [ ]:
high_corr_features = set()
for i in range(len(num1.columns)):
    for j in range(i):
        if abs(num1.iloc[i, j]) > 0.8:
            colname = num1.columns[i]
            high_corr_features.add(colname)

print(f"Features to consider removing: {high_corr_features}")

In [ ]:
train_df = train_df.drop(columns='grade_encoded')
test_df = test_df.drop(columns='grade_encoded')

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

X = train_df.drop(columns='loan_paid_back')
y = train_df['loan_paid_back']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
xgb_model = XGBClassifier()

xgb_model.fit(X_train, y_train)

In [ ]:
y_true = y_test

y_pred = xgb_model.predict(X_test)


from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_true, y_pred, normalize='pred')
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')

# Confusion matrix with different normalization options
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Raw counts
cm_raw = confusion_matrix(y_true, y_pred)
disp_raw = ConfusionMatrixDisplay(confusion_matrix=cm_raw)
disp_raw.plot(ax=ax1, cmap='Blues')
ax1.set_title('Confusion Matrix (Raw Counts)')

# Normalized by true labels (shows recall)
cm_norm = confusion_matrix(y_true, y_pred, normalize='true')
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm)
disp_norm.plot(ax=ax2, cmap='Blues')
ax2.set_title('Confusion Matrix (Normalized by True)')

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# Basic metrics
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1-Score:", f1_score(y_true, y_pred))

# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

In [ ]:
X_submission = test_df.copy()

test_pred = xgb_model.predict(test_df)

submission_df = pd.DataFrame({
    "id": test_id_placeholder,
    "loan_paid_back": test_pred
})

submission_df.to_csv('submission6.csv',index=False)
print("Success!")